In [ ]:
%pip install -q transformers==4.44.2 peft==0.12.0 bitsandbytes==0.43.3 datasets accelerate chromadb sentence-transformers evaluate rouge_score bert_score nltk textstat pandas scikit-learn tqdm


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

BASE_DIR = "/content/drive/MyDrive/Clinical_NLP_Project"
ADAPTER_PATH = f"{BASE_DIR}/adapters/10K"
CHROMA_PATH = "/content/radiology_kb/chroma_db"
PREDICTION_DIR = f"{BASE_DIR}/predictions/10K"
RESULT_DIR = f"{BASE_DIR}/results/10K"

os.makedirs(PREDICTION_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

assert os.path.isdir(ADAPTER_PATH), f"10K adapter folder not found: {ADAPTER_PATH}"
assert os.path.isdir(CHROMA_PATH), f"Chroma database folder not found: {CHROMA_PATH}"
assert torch.cuda.is_available(), "A Colab GPU runtime is required for 4-bit bitsandbytes model loading."

print("Adapter:", ADAPTER_PATH)
print("Chroma:", CHROMA_PATH)
print("Predictions:", PREDICTION_DIR)
print("Results:", RESULT_DIR)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from datasets import load_dataset

DATASET_NAME = "BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track"
SEED = 42

dataset = load_dataset(DATASET_NAME)
shuffled = dataset["train"].shuffle(seed=SEED)
test_data = shuffled.select(range(10100, 10200))

assert len(test_data) == 100

print("Test samples:", len(test_data))
print("Test indices:", 10100, "to", 10199)
print("Columns:", test_data.column_names)


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "biolaysumm_rrg_terms"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name=COLLECTION_NAME)

collection_count = collection.count()
assert collection_count > 0, "The ChromaDB collection is empty."

collection_metadata = collection.metadata or {}
collection_space = collection_metadata.get("hnsw:space", "l2")
sample_records = collection.peek(limit=min(3, collection_count))
sample_documents = sample_records.get("documents") or []
assert sample_documents and all(isinstance(doc, str) and doc.strip() for doc in sample_documents), "The ChromaDB collection does not contain usable text documents."

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Collection loaded:", COLLECTION_NAME)
print("Knowledge entries:", collection_count)
print("Chroma distance space:", collection_space)


In [ ]:
import re

def retrieve_knowledge(report, top_k=4, threshold=0.75):
    sentences = re.split(r'(?<=[.!?])\s+', report)
    selected = []
    seen = set()

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        exact_results = collection.get(
            where_document={"$contains": sentence},
            limit=top_k
        )

        if exact_results and exact_results.get("documents"):
            for doc in exact_results["documents"]:
                if doc and doc not in seen:
                    selected.append(doc)
                    seen.add(doc)

        if len(selected) >= top_k:
            break

    if not selected:
        query_embedding = embedding_model.encode(
            report,
            normalize_embeddings=True
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=min(2, collection.count()),
            include=["documents", "embeddings", "distances"]
        )

        if results and results.get("documents"):
            documents = results["documents"][0]
            document_embeddings = results.get("embeddings")

            if document_embeddings is None or len(document_embeddings) == 0:
                raise RuntimeError("ChromaDB did not return document embeddings; cosine similarity cannot be verified.")

            query_vector = np.asarray(query_embedding, dtype=np.float32)
            query_norm = np.linalg.norm(query_vector)

            for doc, document_vector in zip(documents, document_embeddings[0]):
                if not doc or doc in seen:
                    continue

                document_vector = np.asarray(document_vector, dtype=np.float32)
                denominator = query_norm * np.linalg.norm(document_vector)
                similarity = float(np.dot(query_vector, document_vector) / denominator) if denominator else 0.0

                if similarity >= threshold:
                    selected.append(doc)
                    seen.add(doc)

    return selected[:top_k]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("10K fine-tuned model loaded")


In [ ]:
def create_prompt(report, knowledge):
    knowledge_text = "\n".join([f"- {item}" for item in knowledge])

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You simplify radiology reports for ordinary patients.

Use the retrieved medical knowledge only to explain technical terms.
Do not add findings that are not supported by the original radiology report.

Follow these rules:
1. Preserve the original findings.
2. Preserve negation and uncertainty.
3. Preserve the anatomical location.
4. Do not invent diagnoses.
5. Do not add findings from retrieved knowledge.
6. Do not remove important findings.
7. Explain technical terms in simple language.
8. Avoid unnecessary medical terminology.
9. Do not change the meaning of the report.
10. Do not exaggerate the severity of findings.
11. Use clear and natural language.
12. Produce one concise patient-friendly sentence.

<|eot_id|><|start_header_id|>user<|end_header_id|>

Radiology Report:
{report}

Retrieved Medical Knowledge:
{knowledge_text}

Write the patient-friendly summary.

<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    return prompt


In [ ]:
import pandas as pd
from tqdm.auto import tqdm

predictions = []

for index, sample in enumerate(tqdm(test_data, total=len(test_data), desc="10K RAG predictions")):
    report = str(sample["radiology_report"])
    reference = str(sample["layman_report"])
    knowledge = retrieve_knowledge(report)
    prompt = create_prompt(report, knowledge)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=70,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    predictions.append({
        "validation_index": 10100 + index,
        "radiology_report": report,
        "reference_summary": reference,
        "retrieved_knowledge": " | ".join(knowledge),
        "generated_summary": generated_text
    })

prediction_df = pd.DataFrame(predictions)
prediction_path = os.path.join(PREDICTION_DIR, "finetuned_10K_rag_100_predictions.csv")
prediction_df.to_csv(prediction_path, index=False)

assert len(prediction_df) == 100
print("Predictions saved:", prediction_path)
print("Total predictions:", len(prediction_df))


In [ ]:
df = pd.read_csv(prediction_path)

expected_indices = list(range(10100, 10200))
actual_indices = df["validation_index"].astype(int).tolist()
assert len(df) == 100, f"Expected 100 prediction rows, found {len(df)}"
assert actual_indices == expected_indices, "Prediction indices do not match the selected test samples."

references = df["reference_summary"].fillna("").astype(str).tolist()
generated = df["generated_summary"].fillna("").astype(str).tolist()
reports = df["radiology_report"].fillna("").astype(str).tolist()

assert len(references) == len(generated) == len(reports) == 100
print("Evaluation samples:", len(df))


In [ ]:
import evaluate
import nltk
import numpy as np
import textstat
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer

nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

bleu_result = bleu_metric.compute(
    predictions=generated,
    references=[[reference] for reference in references]
)

rouge_result = rouge_metric.compute(
    predictions=generated,
    references=references
)

meteor_result = meteor_metric.compute(
    predictions=generated,
    references=references
)

bertscore_precision_values, bertscore_recall_values, bertscore_f1_values = bertscore(
    generated,
    references,
    model_type="roberta-large",
    lang="en",
    verbose=True
)

bertscore_precision = float(bertscore_precision_values.mean().item())
bertscore_recall = float(bertscore_recall_values.mean().item())
bertscore_f1 = float(bertscore_f1_values.mean().item())

semantic_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
reference_embeddings = semantic_model.encode(references, normalize_embeddings=True)
generated_embeddings = semantic_model.encode(generated, normalize_embeddings=True)
similarities = np.sum(reference_embeddings * generated_embeddings, axis=1)
semantic_similarity = float(np.mean(similarities))

fkgl_scores = [textstat.flesch_kincaid_grade(text) for text in generated]
dale_chall_scores = [textstat.dale_chall_readability_score(text) for text in generated]
coleman_liau_scores = [textstat.coleman_liau_index(text) for text in generated]

fkgl = float(np.mean(fkgl_scores))
dale_chall = float(np.mean(dale_chall_scores))
coleman_liau = float(np.mean(coleman_liau_scores))

metrics = {
    "BLEU": float(bleu_result["bleu"]),
    "ROUGE-1": float(rouge_result["rouge1"]),
    "ROUGE-2": float(rouge_result["rouge2"]),
    "ROUGE-L": float(rouge_result["rougeL"]),
    "METEOR": float(meteor_result["meteor"]),
    "BERTScore_Precision": bertscore_precision,
    "BERTScore_Recall": bertscore_recall,
    "BERTScore_F1": bertscore_f1,
    "Semantic_Similarity": semantic_similarity,
    "FKGL": fkgl,
    "Dale_Chall": dale_chall,
    "Coleman_Liau": coleman_liau,
    "Evaluation_Samples": len(df)
}

print(metrics)


In [ ]:
from transformers import pipeline

nli_model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
nli_pipeline = pipeline(
    "text-classification",
    model=nli_model_name,
    tokenizer=nli_model_name,
    device=0 if torch.cuda.is_available() else -1
)

nli_id2label = {
    int(key): str(value).lower()
    for key, value in nli_pipeline.model.config.id2label.items()
}

def normalize_nli_label(label):
    normalized = str(label).strip().lower().replace("_", " ").replace("-", " ")
    if normalized.startswith("label "):
        label_id = int(normalized.split()[-1])
        normalized = nli_id2label.get(label_id, normalized)
    normalized = normalized.replace("_", " ").replace("-", " ")
    if "contradiction" in normalized:
        return "contradiction"
    if "entailment" in normalized or normalized == "entail":
        return "entailment"
    if "neutral" in normalized:
        return "neutral"
    raise ValueError(f"Unrecognized NLI label: {label}")

nli_results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="NLI evaluation"):
    result = nli_pipeline(
        {
            "text": str(row["radiology_report"]),
            "text_pair": str(row["generated_summary"])
        },
        truncation=True
    )

    result_item = result[0] if isinstance(result, list) else result
    nli_label = normalize_nli_label(result_item["label"])

    nli_results.append({
        "validation_index": int(row["validation_index"]),
        "nli_label": nli_label,
        "nli_score": float(result_item["score"])
    })

nli_df = pd.DataFrame(nli_results)
assert len(nli_df) == len(df) == 100

label_counts = nli_df["nli_label"].value_counts().to_dict()
entailment_count = int(label_counts.get("entailment", 0))
neutral_count = int(label_counts.get("neutral", 0))
contradiction_count = int(label_counts.get("contradiction", 0))
total = len(nli_df)

assert entailment_count + neutral_count + contradiction_count == total

nli_metrics = {
    "Entailment_Count": entailment_count,
    "Neutral_Count": neutral_count,
    "Contradiction_Count": contradiction_count,
    "Entailment_Percentage": entailment_count / total * 100,
    "Neutral_Percentage": neutral_count / total * 100,
    "Contradiction_Percentage": contradiction_count / total * 100,
    "NLI_Contradiction_Proxy_Percentage": contradiction_count / total * 100,
    "NLI_Model": nli_model_name,
    "NLI_Evaluation_Samples": total
}

print(nli_metrics)


In [ ]:
nli_path = os.path.join(RESULT_DIR, "finetuned_rag_10K_100_results_with_nli.csv")
nli_df.to_csv(nli_path, index=False)

final_metrics = {
    **metrics,
    **nli_metrics
}

metrics_path = os.path.join(RESULT_DIR, "finetuned_rag_10K_complete_metrics.csv")
metrics_json_path = os.path.join(RESULT_DIR, "finetuned_rag_10K_complete_metrics.json")

final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df.to_csv(metrics_path, index=False)

with open(metrics_json_path, "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=4)

print("Final evaluation completed")
print("Metrics CSV:", metrics_path)
print("Metrics JSON:", metrics_json_path)
print("NLI results:", nli_path)
print("Prediction file:", prediction_path)
